# 03_budget_recommendation_logic.

- Done By: Kedar Raju Pawar
- date: 2026-02-26 time: 12:35:00

In [30]:
import pandas as pd

In [31]:
df=pd.read_csv("../../02_Data_Management/04_Processed_Data/analytics_dataset.csv")
df.columns
# Load scaler
import joblib
scaler = joblib.load("performance_scaler.pkl")

## Budget Filter Logic

In [32]:
def recommend_by_budget(budget):
    filtered = df[df["Price"] <= budget]
    return filtered.sort_values("Performance_Score", ascending=False).head(5)

recommend_by_budget(70000) #Show best performance under ₹70K

,Company,laptop_type,screen_size_inch,display_resolution_width,display_resolution_height,panel_type,is_touchscreen,resolution_class,cpu_brand,cpu_family,...,hdd_gb,flash_gb,hybrid_gb,total_storage_gb,weight_kg,weight_class,price_band,Price,is_price_outlier,Performance_Score
901,ASUS,Gaming,17.3,1920,1080,IPS,0,Full HD,Intel,i7,...,0,0,0,512,3.58,Heavy,Mid-Range,68145.1200,No,0.850000
494,ASUS,Gaming,17.3,1920,1080,IPS,1,Full HD,Intel,i7,...,1024,0,0,1280,4.33,Heavy,Mid-Range,67612.3200,No,0.771667
47,ASUS,Gaming,17.3,1920,1080,Standard,0,Full HD,AMD,Ryzen,...,1024,0,0,1280,3.20,Heavy,Mid-Range,69210.7200,No,0.765000
945,LENOVO,Gaming,15.6,1920,1080,IPS,0,Full HD,AMD,FX,...,1024,0,0,1536,2.50,Heavy,Mid-Range,53226.7200,No,0.743333
150,DELL,Gaming,15.6,1920,1080,Standard,0,Full HD,Intel,i7,...,0,0,0,512,2.56,Heavy,Mid-Range,66560.5728,No,0.743333


## What-If Simulation

In [33]:
def simulate_ram_upgrade(index, new_ram):
    temp = df.iloc[index].copy()
    
    temp["ram_gb"] = new_ram
    
    temp_features = temp[features].values.reshape(1, -1)
    scaled = scaler.transform(temp_features)
    
    perf_score = (
        scaled[0][features.index("cpu_tier")] * weights["cpu_tier"] +
        scaled[0][features.index("clock_speed_ghz")] * weights["clock_speed_ghz"] +
        scaled[0][features.index("ram_gb")] * weights["ram_gb"] +
        scaled[0][features.index("ssd_gb")] * weights["ssd_gb"]
    )
    
    return perf_score

In [34]:
df[['ram_gb', 'Price']].head()

,ram_gb,Price
0,8,71378.6832
1,8,47895.5232
2,8,30636.0000
3,16,135195.3360
4,8,96095.8080


In [36]:
# Define performance features again
features = ["ram_gb", "ssd_gb", "cpu_tier", "clock_speed_ghz"]

# Define weights again
weights = {
    "cpu_tier": 0.4,
    "clock_speed_ghz": 0.3,
    "ram_gb": 0.2,
    "ssd_gb": 0.1
}


In [57]:
index = 0
ram_to_upgrade=64

original = df.iloc[index]["Performance_Score"]
upgraded = simulate_ram_upgrade(index, ram_to_upgrade)
print("original cost",df.iloc[index]['Price'])
print("Original:", round(original,3))
print("After Upgrade:", round(upgraded,3))

iimprovement = upgraded - original

avg_ratio = (df["Price"] / df["Performance_Score"]).mean()

rupee_value_of_upgrade = improvement * avg_ratio

print("Improvement:", round(improvement,3))
print(f"Estimated ₹ value of upgrade: ₹{rupee_value_of_upgrade:,.2f}")

original cost 71378.6832
Original: 0.552
After Upgrade: 0.926
Improvement: 0.373
Estimated ₹ value of upgrade: ₹42,213.13


C:\Users\pawar\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but MinMaxScaler was fitted with feature names
  warnings.warn(


In [58]:
from sklearn.linear_model import LinearRegression

X = df[["Performance_Score"]]
y = df["Price"]

model = LinearRegression()
model.fit(X, y)

,fit_intercept,True
,copy_X,True
,tol,1e-06
,n_jobs,None
,positive,False


In [59]:


print(f"Predicted original price: ₹{pred_original:,.2f}")
print(f"Predicted upgraded total price: ₹{pred_upgraded:,.2f}")
print(f"Extra amount to pay: ₹{pred_upgraded - pred_original:,.2f}")

Predicted original price: ₹63,314.85
Predicted upgraded total price: ₹118,646.05
Extra amount to pay: ₹55,331.20
